## DARWIN Sequence Purchasing Problem

Lead     : `<Alex / AlexLeonardos>`

Issue    : [Github Issue #85](https://github.com/petadex/igem-toronto/issues/) — _DARWIN Downstream Oligo Algorithm_

Start    : `2026-08-16`

Script: `resources\260816_issue85_darwinoligo\darwin_oligo.py`

# Description

Oscar has described the DARWIN algorithm for finding the Top-K most influential mutations informed by existing activity data. This will likely be used to generate the first set of sequences for purchasing. Therefore a much simpler version of this sequence purchasing problem will be implemented if the DARWIN algorithm is effective.

This algorithm will simply incorporate the top K mutations, adding the proper degenerate codons for substitution mutations, and fragmentation for addition/deletion mutations.


# Input: 

1. **Base Sequence:** This is the sequence upon which the mutations will be added.
2. **Top-K Mutation List:** This is the actual mutations output by the DARWIN algorithm, which we want to test all of the combinations of. Note that the mutations are 1-indexed, meaning that A2N means that the second position will be swapped from A to N.

Note that the junction method is NOT an input to this algorithm, it will assume the Golden Gate technique is used. This means:

    1. **CGTCTC** and **GAGACG** are BANNED from occuring ANYWHERE in the sequence.
    2. no internal CGGA or GGTG overhangs unless ur doing shared-overhang-minimal-plasmid stuff and want to exclude the first/last fragment(the backbone uses those two overhangs) 


# Output: 

   1. The sequence(s) to order - this is in nucleotide bases.
   2. Indication of which positions have degen codons, fragments.
   3. Any mutations from the top-k list that were impossible to include. This could be due to the lack of a proper junction.

Note that this is the same as the standard algorithm, but we don't need to worry about junk being output. It is recognized that some mutations may require multiple degenerate bases, such as D2K, and this would introduce some junk. In this case, the algorithm always picks the covering codon with the fewest extra amino acids.

# Important Operations

These are the same operations as the standard algorithm, but are used differently. We aren't searching the possible mutation space, we are explicitly given mutations that should be implemented using these operations.

1. **Degenerate Codons**: When at a certain index, there are several possibilities for nucleotide bases. Represents substitution mutations.
2. **Fragments**: Multiple shorter fragments can be ligated together by specific junctions. This allows for consideration of library sequences of different lengths, or for protein differences that can't be efficiently encoded for using degenerate codons. This could be used to deal with addition/deletion mutations.

# Implementation Details:

The algorithm takes the mutations and realizes them depending on their type. Note that each mutation covers both the base input sequence and the mutation, meaning that all $2^K$ combinations of the K realized mutations could be tested:

1. Substitutions: degenerate codons between the wild-type and the mutant.
2. Indels: fragment variants (the fragment is ordered twice, with and without), one fragment per indel. This is because a length change can't be encoded by a single oligo.
3. Unencodable substitutions: demoted to fragment variants rather than dropped. This is more expensive, but it was chosen to ensure that all mutations were included if possible.

Anything that can't be realized is reported with a reason in the output. This could occur if multiple split sites for Golden-Gate Assembly are too close to each other, or there isn't a valid split site in general.

Note that the Golden-Gate Assembly conditions are met using the same restrictions as the standard algorithm. A standard search over the possible junctions is done, excluding the banned junctions, and the chosen overhangs for the different mutations must be mutually orthogonal (all must be different in terms of identity and reverse complement). Additionally, after every codon choice, a sliding window is passed across the IUPAC oligos to check for the banned sequences as well.

## Test

The following is an LLM-generated test for the algorithm. This will be further tested with actual example outputs from the DARWIN algorithm in the future.

# Results (August 16th, 2026)

The test case was chosen to exercise **every path at once**: ordinary substitutions, an insertion, a deletion, a substitution that cannot be encoded by any degenerate codon, and two malformed inputs.

## Input 1 — Base Sequence

A single PETase core (290 aa) — `cluster2`'s first core with gaps removed:

```
>darwin_base
GAADRGGMQHMSTSIARVRTRLAALVAGVVVAGSTVIGASPAGAQENPYERGPDPTESSI
EAVRGPFTVAQTSVSRLAANGFGGGTIYYPTDTSQGTFGAVAISPGFTAGESSIAWLGPR
IASQGFVVITIDTLTRLDQPDSRGRQLQAALDHLRTNSTVRNRIDPNRMAVMGHSMGGGG
ALSAAASNTDLEAAIPLQGWHTRKNWSSVRTPTLVIGAQLDSIAPVGSHSEAFYESLPSD
LDKAYMELRGASHFVSNTPDTTTAKYSIAWLKRFVDDDLRYEQFLCPAPD
```

## Input 2 — Mutation List

```
V29A    substitution
S40T    substitution
R120K   substitution
-74Q    insertion  (insert Q immediately before residue 74)
V171-   deletion   (delete the V at residue 171)
W200K   substitution, deliberately unencodable
X5Y     deliberately wrong: the base sequence has R at position 5
A2      deliberately malformed: no destination residue
```

Command run. The `--mutations="..."` form is required — an insertion token begins with `-`, which a bare argument list would read as a flag:

```bash
python darwin_oligo.py --seq-file base.fasta \
    --mutations="V29A,S40T,R120K,-74Q,V171-,W200K,X5Y,A2"
```

## Output

```
base sequence: 290 aa
mutations given: 8   chemistry: Golden Gate (BSMBI, site CGTCTC)

MUTATIONS
  label      type         status    how / why
  V29A       sub          included  degenerate codon GYA at residue 29 (encodes AV)
  S40T       sub          included  degenerate codon ASC at residue 40 (encodes ST)
  R120K      sub          included  degenerate codon ARA at residue 120 (encodes KR)
  -74Q       ins          included  fragment variant (2 oligos: with and without) at residue 74
  V171-      del          included  fragment variant (2 oligos: with and without) at residue 171
  W200K      sub          included  fragment variant (2 oligos: with and without) at residue 200
  X5Y        sub          dropped   base sequence has R at position 5, not X
  A2         sub          dropped   unparseable (expected forms A2N, -4R, G5-)
  -> 6/8 realised

DESIGN: 3 fragment(s)
  fragment 1: residues 1-75 (75 aa), 2 oligo(s), 453 nt
      - wild-type                       4 variant(s), 225 nt
      - -74Q                            4 variant(s), 228 nt
      degenerate codons at residue(s): 29, 40
  fragment 2: residues 76-172 (97 aa), 2 oligo(s), 579 nt
      - wild-type                       2 variant(s), 291 nt
      - V171-                           2 variant(s), 288 nt
      degenerate codons at residue(s): 120
  fragment 3: residues 173-290 (118 aa), 2 oligo(s), 708 nt
      - wild-type                       1 variant(s), 354 nt
      - W200K                           1 variant(s), 354 nt

JUNCTIONS (Golden Gate) -- individually high-fidelity, mutually orthogonal:
  before residue 76: overhang 5'-CTCG-3' (rc CGAG), pinned codons TCT|CGT  [S75|R76]
  before residue 173: overhang 5'-TGGG-3' (rc CCCA), pinned codons ATG|GGT  [M172|G173]
  [backbone overhangs CGGA | GGTG reserved -- excluded from internal junctions]

ORDER
  oligos to order : 6
  total synthesis : 1,740 nt
  proteins produced (all combinations): 64

forbidden-site check on 4 assembled full-length example(s): clean
```

Each run is saved to `darwinruns/<timestamp>_<stem>_darwin/` containing `report.txt`, `summary.json`, `oligos.fasta` (the six sequences to order) and `examples_full_length_dna.fasta`.

## What the run confirms

**1. The library is genuinely combinatorial.** 64 proteins = $2^6$, exactly all combinations of the six realized mutations — including the unmutated parent, because every substitution codon encodes the wild-type residue as well as the mutant. The arithmetic decomposes across fragments as $8 \times 4 \times 2$: fragment 1 contributes 2 oligos × 2 substitutions, fragment 2 contributes 2 oligos × 1 substitution, fragment 3 contributes 2 oligos.

**2. No junk was introduced by the degenerate codons.** All three substitutions had an exact covering codon — `GYA` encodes exactly {A,V}, `ASC` exactly {S,T}, `ARA` exactly {K,R}. This is the good case; the `D2K` situation described above would instead have shown extra amino acids in the `encodes` column.

**3. The fallback path fired on a real case.** `W200K` is unencodable: every degenerate codon covering both W and K also admits a stop codon (`TAG`/`TAA`), which would truncate the protein. Rather than dropping an influential mutation, it was demoted to a fragment variant — 2 oligos for fragment 3 instead of 1.

**4. Both input-validation drops behaved as intended.** `X5Y` was checked against the base sequence and rejected with the residue actually found there; `A2` failed to parse. Neither silently corrupted the design.

**5. The Golden Gate constraints hold.** The two overhangs `CTCG` and `TGGG` are non-palindromic, GC-balanced, mutually orthogonal, and distinct from the reserved backbone overhangs. Both sit on unmutated residues (S75|R76 and M172|G173), so their codons could be pinned. Four assembled full-length sequences were checked and contain no `CGTCTC`/`GAGACG`.

## Separate check — the "impossible to include" path

The run above has no junction failures, so that path was exercised separately with two **adjacent** deletions:

```bash
python darwin_oligo.py --seq-file base.fasta --mutations="A78-,A79-,V171-"
```

```
  A78-       del          included  fragment variant (2 oligos: with and without) at residue 78
  A79-       del          dropped   no orthogonal Golden Gate junction available to isolate it
  V171-      del          included  fragment variant (2 oligos: with and without) at residue 171
  -> 2/3 realised
```

Isolating both deletions in their own fragments would require a cut between residues 78 and 79, but a junction pins the codons of the residues on either side of the cut, and both of those residues are themselves deleted in one variant. There is nowhere legal to cut, so the second deletion is reported rather than being silently merged or dropped without explanation.
